# Template 07: SHAP Analysis

**Purpose:** Aggregate SHAP values and generate feature analysis plots

**Inputs:**
- results/06_shap_test.parquet
- data/04_test.parquet

**Outputs:**
- results/07_shap_aggregate.csv
- results/07_plots/ (residual + range plots for each feature)

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import numpy as np
import yaml
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
from shap_utils import compute_shap_aggregate, create_residual_plot, create_shap_range_plot, create_feature_importance_plot

print("########################################")
print("# STAGE 07: SHAP ANALYSIS")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
os.makedirs(f"{output_base}/results/07_plots", exist_ok=True)

In [ ]:
# Load SHAP dataframe from Template 06
shap_test = pd.read_parquet(f"{output_base}/results/06_shap_test.parquet")
print(f"\n* SHAP data loaded: {shap_test.shape}")

# Load test data from Template 04
test_data = pd.read_parquet(f"{output_base}/data/04_test.parquet")
print(f"* Test data loaded: {test_data.shape}")

# Add weight column (use vin as proxy if no actual weight)
if 'weight' not in shap_test.columns:
    shap_test['weight'] = 1
    print(f"  Added unit weights")

## Step 1: Aggregate SHAP Values

In [ ]:
# Compute SHAP aggregates (no globals!)
print(f"\n* Computing SHAP aggregates...")
shagg, shagg_num, shagg2 = compute_shap_aggregate(shap_test, weight_col='weight')

print(f"  Total features: {len(shagg)}")
print(f"  Features with SHAP > 0: {len(shagg2)}")

# Save aggregate results
shagg_file = f"{output_base}/results/07_shap_aggregate.csv"
shagg_num.to_csv(shagg_file, index=False)
print(f"  Saved: {shagg_file}")

print(f"\nTop 10 features by SHAP importance:")
print(shagg_num[['field', 'total_shap', 'shap_abs_pct', 'cum_shap_abs_pct']].head(10))

## Step 2: Feature Importance Plot

In [ ]:
# Create feature importance plot
print(f"\n* Generating feature importance plot...")
fig_importance = create_feature_importance_plot(shagg_num, top_n=20)

importance_file = f"{output_base}/results/07_feature_importance.png"
fig_importance.savefig(importance_file, dpi=150, bbox_inches='tight')
plt.close(fig_importance)

print(f"  Saved: {importance_file}")

## Step 3: Per-Feature Analysis (Loop)

In [ ]:
# Prepare test data for residual plots
test_data['incurred_act'] = test_data[cfg['experiment']['target']]
test_data['incurred_pred'] = test_data['pred']
test_data['denom'] = 1  # Adjust if using exposures
weight_col = 'vin'  # Adjust if actual weight column exists

print(f"\n* Prepared data for feature analysis")

In [ ]:
# Loop through features and create plots
print(f"\n* Generating per-feature plots...")
print(f"  Processing {len(shagg2)} features")

feature_list = shagg2['field'].tolist()
plots_generated = 0

for i, feature in enumerate(feature_list[:10]):  # Top 10 for demo, remove [:10] for all
    try:
        # Check if feature exists in both datasets
        if feature not in test_data.columns or feature not in shap_test.columns:
            print(f"  [{i+1}/{len(feature_list)}] Skipping {feature} - not in data")
            continue
        
        # Residual plot
        fig_resid, _ = create_residual_plot(
            test_data, 
            feature, 
            weight_col, 
            print_table=False
        )
        resid_file = f"{output_base}/results/07_plots/{feature}_residual.png"
        fig_resid.savefig(resid_file, dpi=100, bbox_inches='tight')
        plt.close(fig_resid)
        
        # SHAP range plot
        fig_range, _ = create_shap_range_plot(
            shap_test,
            test_data,
            feature,
            weight_col,
            shap_round_level=3,
            feature_round_to=0.001,
            min_ntile=5,
            max_ntile=95,
            filter_used_only=False
        )
        range_file = f"{output_base}/results/07_plots/{feature}_shap_range.png"
        fig_range.savefig(range_file, dpi=100, bbox_inches='tight')
        plt.close(fig_range)
        
        plots_generated += 2
        print(f"  [{i+1}/{len(feature_list)}] {feature}: ✓")
        
    except Exception as e:
        print(f"  [{i+1}/{len(feature_list)}] {feature}: ERROR - {str(e)}")

print(f"\n* Generated {plots_generated} plots for {plots_generated//2} features")
print(f"  Location: {output_base}/results/07_plots/")

In [ ]:
print("\n########################################")
print("# STAGE 07: COMPLETE")
print("########################################")